# Sentiment Analysis of Twitter Posts
<!-- Notebook name goes here -->
<center><b>Notebook: Model Evaluations and Conclusions</b></center>
<br>

**By**: Stephen Borja, Justin Ching, Erin Chua, and Zhean Ganituen.

**Dataset**: Hussein, S. (2021). Twitter Sentiments Dataset [Dataset]. Mendeley. https://doi.org/10.17632/Z9ZW7NT5H2.1

**Motivation**: Every minute, social media users generate a large influx of textual data on live events. Performing sentiment analysis on this data provides a real-time view of public perception, enabling quick insights into the general population’s opinions and reactions.

**Goal**: By the end of the project, our goal is to create and compare supervised learning algorithms for sentiment analysis.

# **1. Project Setup**

We first import the relevant libraries for the three models.


In [ ]:
# PyTorch
import torch
import torch.nn as nn

# General Imports
import sys, os

sys.path.append(os.path.abspath("../lib"))

from anal_ysis_tools import (
    confusion_matrix,
    comparative_confusion_matrix,
    incorrect_confidence_score,
)

# Select device (using CUDA if possible)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", DEVICE)

# **2. Data Setup**

Now, we run the data processing pipeline. The below cell's sole purpose is to run the `data.ipynb` notebook, which runs everything related to data cleaning and data pre-processing.

In [ ]:
import IPython.core.page
import builtins
from IPython.utils.capture import capture_output

pager = IPython.core.page.page
helper = builtins.help

IPython.core.page.page = lambda *args, **kwargs: None
builtins.help = lambda *args, **kwargs: None

try:
    with capture_output():
        %run data.ipynb
finally:
    IPython.core.page.page = pager
    builtins.help = helper

print("Data Setup is DONE")

# Tests
assert X.shape == (162_801, 29318), "Feature matrix shape is wrong; expected (162_801, 29318)"
assert y.shape == (162_801,), "Labels shape is wrong; expected (162_801,)"

assert X_train.shape == (113_960, 29_318), "Train shape is wrong; expected (113_960, 29318)"
assert Xb_train.shape == (74_565, 29_318), "Train shape is wrong; expected (74_292, 29318)"
assert X_val.shape == (24_420, 29_318), "Validation shape is wrong; expected (24_420, 29318)"
assert X_test.shape == (24_421, 29_318), "Test shape is wrong; expected (24_421, 29318)"

assert y_train.shape == (113_960,), "Train labels shape is wrong; expected (113_960,)"
assert yb_train.shape == (74_565,), "Train labels shape is wrong; expected (74_292,)"
assert y_val.shape == (24_420,), "Validation labels shape is wrong; expected (24_420,)"
assert y_test.shape == (24_421,), "Test labels shape is wrong; expected (24_421,)"
print("All tests passed.")

For our neural network, we prepare the dataset to be PyTorch-compatible.

Because our labels in `y_train` are `-1`, `0`, and `1`, they are incompatible with PyTorch which requires class labels to be non-negative integers. TorchableSet maps them to non-negative integers so that PyTorch can handle them.

TorchableSet also ensures that the features (`X_train`) are stored in a sparse matrix and the labels (`y_train`) are in a NumPy array for easier handling.

In [ ]:
from barn.data_preparer import TorchableSet

TRAIN_IMB = TorchableSet(X_train, y_train)
TRAIN_BAL = TorchableSet(Xb_train, yb_train)
TEST = TorchableSet(X_test, y_test)
VAL = TorchableSet(X_val, y_val)

We also need to set up our neural network architecture.

In [ ]:
class MyLittlePony(nn.Module):
    def __init__(self, vocab_size, neurons_in_hidden, n_hidden_layers, dropout):
        """
        Architecture definition for the neural network.

        # Parameters
        * vocab_size: number of words in the dataset, this is the number of neurons in the input layer
        * neurons_in_hidden: number of neurons in the hidden layers
        * n_hidden_layers: number of hidden layers
        * dropout: dropout value

        # Remarks
        * Documenation is provided for each layer. Notation is as follows
            * n is the number of neurons
            * a is the activation function
        * Number of neurons in the output layer is a constant 3 (one for each sentiment in the dataset).
        """
        super().__init__()
        # INPUT LAYER
        # -----------
        # n  :: vocab_size
        # a  :: ReLU
        layers = [
            nn.Linear(vocab_size, neurons_in_hidden),
            nn.ReLU(),
            nn.Dropout(dropout),
        ]

        # HIDDEN LAYER
        # ------------
        # NN :: hidden_dim
        # A  :: ReLU
        for _ in range(n_hidden_layers - 1):
            layers.append(nn.Linear(neurons_in_hidden, neurons_in_hidden))
            layers.append(nn.ReLU())
            layers.append(nn.Dropout(dropout))

        # OUTPUT LAYER
        # ------------
        # n  :: 3    (for each sentiment)
        # a  :: none (left as XΘ)
        layers.append(nn.Linear(neurons_in_hidden, 3))

        self.model = nn.Sequential(*layers)
        self._initialize_weights()

    def _initialize_weights(self):
        """
        Kaiming He initialization
        """
        for module in self.modules():
            if isinstance(module, nn.Linear):
                nn.init.kaiming_uniform_(module.weight, nonlinearity="relu")
                if module.bias is not None:
                    nn.init.zeros_(module.bias)

    def forward(self, x):
        return self.model(x)

    def predict_proba(self, X):
        self.eval()
        device = next(self.parameters()).device
        with torch.no_grad():
            X = X.to(device)
            logits = self(X)

            probabilities = torch.softmax(logits, dim=1)
            return probabilities.cpu().numpy()

Then, we'll grab our trained models from the other notebooks.